In [0]:
from pyspark.sql import Row

row1 = Row(name='Alice', age=14, subject='Math', grade='B')
row2 = Row(name='Lucas', age=14, subject='Biology', grade='A')

In [0]:
df = spark.createDataFrame([row1, row2])

df

In [0]:
df.show()

In [0]:
df.take(1)

In [0]:
display(df.take(1))

In [0]:
display(df)

In [0]:
students = [{"name": "Alice", "age": 14, "subject": "Math", "grade": "B"}, 
            {"name": "Lucas", "age": 14, "subject": "Biology", "grade": "A"}]
    
students_df = spark.createDataFrame(students)
students_df

In [0]:
display(students_df)

In [0]:
students_df.take(2)

In [0]:
dbutils.fs.ls("file:/Workspace/Users/kalle.rydberg02@gmail.com/databricks_karl_de25/pyspark_intro")

In [0]:
import os
import pandas as pd

# Get notebook folder dynamically
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = "/Workspace" + os.path.dirname(notebook_path)

# Read with pandas (uses local filesystem, no URI prefix needed)
pandas_df = pd.read_csv(f"{base_path}/data/athlete_events.csv")

# Convert to Spark dataframe
athlete_df = spark.createDataFrame(pandas_df)
display(athlete_df.take(5))

In [0]:
athlete_df.printSchema()

In [0]:
athlete_df.columns

In [0]:
from pyspark.sql.types import StructField, StructType, ShortType, StringType, IntegerType, DoubleType, ByteType

schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", ByteType(), True),
    StructField("Height", ShortType(), True),
    StructField("Weight", ShortType(), True),
    StructField("Team", StringType(), True),
    StructField("NOC", StringType(), True),
    StructField("Games", StringType(), True),
    StructField("Year", ShortType(), True),
    StructField("Season", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Sport", StringType(), True),
    StructField("Event", StringType(), True),
    StructField("Medal", StringType(), True)
])

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = "/Workspace" + os.path.dirname(notebook_path)

# Read with pandas, convert with schema
pandas_df = pd.read_csv(f"{base_path}/data/athlete_events.csv")
athlete_df = spark.createDataFrame(pandas_df, schema=schema)

#display(athlete_df.take(5))

athlete_df.describe

In [0]:
from pyspark.sql.functions import col, sum

nulls = athlete_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in athlete_df.columns]
)
display(nulls)

In [0]:
display(athlete_df.filter("NOC = 'SWE'"))

In [0]:
athlete_df.groupBy("NOC").count().filter("NOC IN ('SWE', 'NOR', 'FIN', 'DEN')").show()

In [0]:
import os
import pandas as pd

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = "/Workspace" + os.path.dirname(notebook_path)

pandas_df = pd.read_csv(f"{base_path}/data/athlete_events.csv")
athlete_df = spark.createDataFrame(pandas_df, schema=schema)  # must be spark.createDataFrame, not athlete_df.createDataFrame

print(type(athlete_df))

In [0]:
athlete_df.createOrReplaceTempView("athlete_df")

df_swe_medals = spark.sql("""
                          SELECT
                           sport, count(medal) as medals
                          FROM athlete_df
                          WHERE noc = 'SWE' AND medal IN ('Gold', 'Silver', 'Bronze')
                          GROUP BY sport
                          ORDER by medals DESC""")

display(df_swe_medals)

In [0]:
fig = df_swe_medals.plot(kind='bar', x='medals', y='sport', title='Swedish medals by sport')

fig.update_layout(yaxis= {"autorange": "reversed"})

In [0]:
%sql

FROM athlete_df LIMIT 1

In [0]:
%sql

SHOW CATALOGS;

In [0]:
%sql

DROP CATALOG IF EXISTS data;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS data;

CREATE SCHEMA IF NOT EXISTS data.olympics;

CREATE TABLE IF NOT EXISTS data.olympics.sweden_medals AS
(
  SELECT
    name,
    age,
    weight,
    height,
    year,
    sport,
    medal
  FROM
    athlete_df
  WHERE
    noc = 'SWE'
    and medal IN ('Gold', 'Bronze', 'Silver')
);

In [0]:
%sql

FROM data.olympics.sweden_medals;